In [5]:
# ============================================================
# RESPONSIBLE RAG SYSTEM (FINAL WITH CORRECT EVALUATION)
# ============================================================

from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("\n[INFO] Creating dataset...")

documents = [
    "Artificial Intelligence is a branch of computer science that enables machines to learn and make decisions.",
    "Machine learning is a subset of artificial intelligence focused on data-driven learning.",
    "Deep learning uses neural networks to model complex patterns.",
    "Cybersecurity protects systems and networks from cyber attacks.",
    "Natural language processing allows machines to understand human language."
]

queries = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is cybersecurity?"
]

ground_truth_answers = [
    documents[0],
    documents[1],
    documents[3]
]

relevant_docs = [
    [documents[0]],
    [documents[1]],
    [documents[3]]
]

# -------------------------------
# STEP 3: EMBEDDINGS
# -------------------------------
print("\n[INFO] Creating embeddings...")

embedder = SentenceTransformer('paraphrase-MiniLM-L3-v2', device='cpu')
doc_embeddings = embedder.encode(documents, batch_size=4)

# -------------------------------
# STEP 4: FAISS INDEX
# -------------------------------
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# -------------------------------
# STEP 5: RETRIEVAL
# -------------------------------
def retrieve(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, k)
    docs = [documents[i] for i in indices[0]]
    return docs

# -------------------------------
# STEP 6: RERANKING
# -------------------------------
def rerank(query, docs):
    query_emb = embedder.encode([query])[0]
    doc_embs = embedder.encode(docs)

    scores = cosine_similarity([query_emb], doc_embs)[0]
    sorted_docs = [doc for _, doc in sorted(zip(scores, docs), reverse=True)]

    return sorted_docs[:1]

# -------------------------------
# STEP 7: GENERATOR
# -------------------------------
print("\n[INFO] Loading generator...")

generator = pipeline("text2text-generation", model="google/flan-t5-small")

# -------------------------------
# STEP 8: RAG PIPELINE
# -------------------------------
def rag_pipeline(query):
    docs = retrieve(query, k=3)
    docs = rerank(query, docs)

    context = " ".join(docs)

    print("\n[DEBUG] Context Used:")
    print(context)

    prompt = f"""
    You are a precise AI assistant.

    Use ONLY the information from the context.

    Context:
    {context}

    Question:
    {query}

    Answer in one clear sentence:
    """

    response = generator(prompt, max_length=100)[0]['generated_text']
    return response, docs

# -------------------------------
# STEP 9: RESPONSIBLE AI MODULES (CORRECTED)
# -------------------------------

def safety_filter(text):
    banned = ["hate", "violence"]
    return "BLOCKED" if any(w in text.lower() for w in banned) else text

# ✅ NEW: Groundedness
def groundedness_score(answer, docs):
    context = " ".join(docs)

    ans_emb = embedder.encode([answer])
    ctx_emb = embedder.encode([context])

    similarity = cosine_similarity(ans_emb, ctx_emb)[0][0]
    return similarity

# ✅ NEW: Hallucination = inverse
def hallucination_score(groundedness):
    return 1 - groundedness

def bias_score(text):
    sensitive = ["gender", "religion"]
    return 1 if any(w in text.lower() for w in sensitive) else 0

# -------------------------------
# STEP 10: EVALUATION METRICS
# -------------------------------

def precision_at_k(retrieved, relevant):
    return len(set(retrieved) & set(relevant)) / len(retrieved)

def recall_at_k(retrieved, relevant):
    return len(set(retrieved) & set(relevant)) / len(relevant)

def f1_score(p, r):
    return 2*p*r/(p+r) if (p+r) > 0 else 0

def answer_similarity(ans, gt):
    emb1 = embedder.encode([ans])
    emb2 = embedder.encode([gt])
    return cosine_similarity(emb1, emb2)[0][0]

# -------------------------------
# STEP 11: RUN EXPERIMENT (CORRECTED)
# -------------------------------
print("\n===============================")
print(" RUNNING EXPERIMENT")
print("===============================\n")

results = []

for i, query in enumerate(queries):

    print(f"\n[QUERY] {query}")

    answer, docs = rag_pipeline(query)
    answer = safety_filter(answer)

    p = precision_at_k(docs, relevant_docs[i])
    r = recall_at_k(docs, relevant_docs[i])
    f1 = f1_score(p, r)

    sim = answer_similarity(answer, ground_truth_answers[i])

    # ✅ FIXED PART
    groundedness = groundedness_score(answer, docs)
    hallucination = hallucination_score(groundedness)

    bias = bias_score(answer)

    print("[ANSWER]", answer)
    print("[PRECISION]", p)
    print("[RECALL]", r)
    print("[F1]", f1)
    print("[SIMILARITY]", sim)
    print("[GROUNDEDNESS SCORE]", groundedness)
    print("[HALLUCINATION SCORE]", hallucination)
    print("[BIAS]", bias)

    # ✅ UPDATED STRUCTURE
    results.append([p, r, f1, sim, groundedness, hallucination, bias])

# -------------------------------
# STEP 12: FINAL METRICS (CORRECTED)
# -------------------------------
results = np.array(results)

print("\n===============================")
print(" FINAL RESULTS")
print("===============================")

print("Avg Precision:", results[:,0].mean())
print("Avg Recall:", results[:,1].mean())
print("Avg F1:", results[:,2].mean())
print("Avg Similarity:", results[:,3].mean())

# ✅ CORRECT METRICS
print("Avg Groundedness:", results[:,4].mean())
print("Hallucination Rate:", results[:,5].mean())
print("Bias Rate:", results[:,6].mean())

print("\n===============================")


[INFO] Creating dataset...

[INFO] Creating embeddings...


C:\Users\qawsar.gulzar\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



[INFO] Loading generator...

 RUNNING EXPERIMENT


[QUERY] What is artificial intelligence?

[DEBUG] Context Used:
Artificial Intelligence is a branch of computer science that enables machines to learn and make decisions.
[ANSWER] a branch of computer science that enables machines to learn and make decisions
[PRECISION] 1.0
[RECALL] 1.0
[F1] 1.0
[SIMILARITY] 0.8159268
[GROUNDEDNESS SCORE] 0.8159268
[HALLUCINATION SCORE] 0.18407320976257324
[BIAS] 0

[QUERY] What is machine learning?

[DEBUG] Context Used:
Machine learning is a subset of artificial intelligence focused on data-driven learning.
[ANSWER] a subset of artificial intelligence focused on data-driven learning
[PRECISION] 1.0
[RECALL] 1.0
[F1] 1.0
[SIMILARITY] 0.9061582
[GROUNDEDNESS SCORE] 0.9061582
[HALLUCINATION SCORE] 0.0938417911529541
[BIAS] 0

[QUERY] What is cybersecurity?

[DEBUG] Context Used:
Cybersecurity protects systems and networks from cyber attacks.
[ANSWER] protects systems and networks from cyber attacks
[PR